# 01 — Raw Data Exploration & Profiling
## Brazilian E-Commerce Public Dataset by Olist

This notebook systematically profiles all 9 raw datasets, analyzing schemas, row counts, null distributions, unique constraints, and data anomalies.


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from src.config import RAW_DATA_DIR, RAW_FILES

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)


### 1. Ingest Raw Datasets


In [ ]:
raw_data = {}
for name, filename in RAW_FILES.items():
    path = RAW_DATA_DIR / filename
    df = pd.read_csv(path)
    raw_data[name] = df
    print(f"{name:22s} | Rows: {df.shape[0]:>10,d} | Columns: {df.shape[1]:>2d}")


### 2. Dataset Schema & Missing Value Profiling


In [ ]:
profile_summary = []
for name, df in raw_data.items():
    for col in df.columns:
        profile_summary.append({
            "dataset": name,
            "column": col,
            "dtype": str(df[col].dtype),
            "null_count": int(df[col].isna().sum()),
            "null_pct": round(float(df[col].isna().mean()) * 100, 2),
            "unique_count": int(df[col].nunique())
        })

profile_df = pd.DataFrame(profile_summary)
print(profile_df[profile_df["null_pct"] > 0])


### 3. Orders Lifecycle & Status Distribution


In [ ]:
orders = raw_data["orders"]
print(orders["order_status"].value_counts())
print("
Missing delivery dates on delivered orders:", orders[orders["order_status"] == "delivered"]["order_delivered_customer_date"].isna().sum())


### 4. Financial Ranges & Price Outliers


In [ ]:
items = raw_data["order_items"]
print(items[["price", "freight_value"]].describe())


### 5. Review Score Distribution


In [ ]:
reviews = raw_data["order_reviews"]
print(reviews["review_score"].value_counts().sort_index(ascending=False))
